# Mamba-2 Language Model demo

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import torch
from transformers import AutoTokenizer

from mamba2 import Mamba2LMHeadModel

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

Official pretrained models on [huggingface](https://huggingface.co/state-spaces):
* `state-spaces/mamba2-130m`
* `state-spaces/mamba2-370m`
* `state-spaces/mamba2-780m`
* `state-spaces/mamba2-1.3b`
* `state-spaces/mamba2-2.7b`

Choose a model depending on available system RAM (for CPU or system with unified memory) or VRAM.

Note that these are base models without fine-tuning for downstream tasks such as chat or instruction following.

In [ ]:
model = Mamba2LMHeadModel.from_pretrained("state-spaces/mamba2-130m", device=device) # state-spaces/mamba2-1.3b" , "state-spaces/mamba2-130m"
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
generation_config = dict(
    max_new_length=200,
    temperature=1.0,
    top_k=30,
    top_p=1.0,
)

In [ ]:
def generate(prompt: str, seed: int = 0, show_perf: bool = True):
    """Generate streaming completion"""
    torch.manual_seed(seed)

    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)[0]
    print(prompt, end="")

    start = time.process_time()
    n_generated = 0
    for i, (token_id, _hidden_state) in enumerate(model.generate(input_ids, **generation_config)):
        token = tokenizer.decode([token_id])
        if i == 0:
            now = time.process_time()
            prompt_eval_elapsed, start = now - start, now
        else:
            n_generated += 1
        print(token, end="", flush=True)
    if show_perf:
        elapsed = time.process_time() - start
        print('\n\n---')
        print(f'Prompt eval | tokens: {input_ids.shape[0]} | elapsed: {prompt_eval_elapsed:.2f}s | tok/s: {input_ids.shape[0] / prompt_eval_elapsed:.2f}')
        print(f'Generation | tokens: {n_generated} | elapsed: {elapsed:.2f}s | tok/s: {n_generated / elapsed:.2f}')

In [22]:
generate("Japan is ")

Japan is 
MPC step 0 top-10
1 187 '\n' 17.435684204101562
2 11733 'icing' 17.014972686767578
3 5491 'iced' 14.908531188964844
4 571 'ia' 13.997049331665039
5 603 'ire' 13.870102882385254
6 757 'ian' 13.684148788452148
7 2155 '�' 13.656984329223633
8 461 'ust' 13.393641471862793
9 398 'ers' 13.381879806518555
10 2874 '001' 13.365482330322266


MPC step 1 top-10
1 783 'the' 64.13382720947266
2 29114 'taking' 63.83436584472656
3 13565 'looking' 63.51449966430664
4 23350 'still' 63.29503631591797
5 66 'a' 63.27836227416992
6 47590 'currently' 63.25233459472656
7 249 'in' 63.11475372314453
8 251 'on' 63.04655456542969
9 84 's' 62.9749641418457
10 8190 'contin' 62.9169921875
the
MPC step 2 top-10
1 1533 ' world' -68.93504333496094
2 806 ' first' -69.38406372070312
3 954 ' most' -69.48786926269531
4 6253 ' largest' -69.74299621582031
5 760 ' only' -69.94413757324219
6 1273 ' second' -70.22864532470703
7 2626 ' third' -70.77418518066406
8 1180 ' number' -70.91441345214844
9 2586 ' country' -70

In [ ]:
generate("日本についてどう思う？")

In [ ]:
generate("CUDA is Nvidia's biggest moat")

In [ ]:
import crypten
import torch

for a in range(1,-1000,-10):
    x = torch.ones(1, 1) * -99
    print(torch.exp(x))
    crypten.init()
    x = crypten.cryptensor(x)
    print(x.exp().get_plain_text())

#clamp  ~ 5

In [ ]:
import crypten
import torch

for a in range(1,1500,10):
    x = torch.ones(1, 1) * a
    print("rsqrt plain:",torch.rsqrt(x))
    crypten.init()
    x = crypten.cryptensor(x)
    print("rsqrt crypt:",x.inv_sqrt().get_plain_text())

# clamp  1 ~ 165

In [ ]:
import crypten
import torch

def silu(x):
    return x * torch.sigmoid(x)

def _silu(x):
    return x * x.sigmoid()

for a in range(1,1000,10):
    x = torch.ones(1, 1) * a
    print("silu plain:",silu(x))
    crypten.init()
    x = crypten.cryptensor(x)
    print("silu crypt:",_silu(x).get_plain_text())

# clamp   ~ 500

In [ ]:
import crypten
import torch

def softplus(x):
    one = torch.ones(1, 1)
    return torch.log((torch.exp(x) + 1.0))

def _softplus(x):
    return (x.exp() + 1.0).log()

for a in range(1,100,1):
    x = torch.ones(1, 1) * a
    print("softplus plain:",softplus(x))
    crypten.init()
    x = crypten.cryptensor(x)
    print("softplus crypt:",_softplus(x).get_plain_text())

# clamp  ~ 5

In [ ]:
import crypten
import torch

for a in range(1,1000,10):
    print(f"a is {a}")
    x = torch.ones(1, 1) * a
    print(torch.log(x))
    crypten.init()
    x = crypten.cryptensor(x)
    print(x.log().get_plain_text())
    print("")

#clamp  ~ 260

a is 1
tensor([[0.]])
tensor([[0.0004]])

a is 11
tensor([[2.3979]])
tensor([[2.3880]])

a is 21
tensor([[3.0445]])
tensor([[3.0278]])

a is 31
tensor([[3.4340]])
tensor([[3.4079]])

a is 41
tensor([[3.7136]])


/opt/conda/lib/python3.11/site-packages/crypten/__init__.py:64: RuntimeWarning: CrypTen is already initialized.
  warnings.warn("CrypTen is already initialized.", RuntimeWarning)


tensor([[3.6852]])

a is 51
tensor([[3.9318]])
tensor([[3.9022]])

a is 61
tensor([[4.1109]])
tensor([[4.0751]])

a is 71
tensor([[4.2627]])
tensor([[4.2279]])

a is 81
tensor([[4.3944]])
tensor([[4.3565]])

a is 91
tensor([[4.5109]])
tensor([[4.4720]])

a is 101
tensor([[4.6151]])
tensor([[4.5724]])

a is 111
tensor([[4.7095]])
tensor([[4.6687]])

a is 121
tensor([[4.7958]])
tensor([[4.7552]])

a is 131
tensor([[4.8752]])
tensor([[4.8302]])

a is 141
tensor([[4.9488]])
tensor([[4.9014]])

a is 151
tensor([[5.0173]])
tensor([[4.9693]])

a is 161
tensor([[5.0814]])
tensor([[5.0327]])

a is 171
tensor([[5.1417]])
tensor([[5.0905]])

a is 181
tensor([[5.1985]])
tensor([[5.1431]])

a is 191
tensor([[5.2523]])
tensor([[5.1973]])

a is 201
tensor([[5.3033]])
tensor([[5.2451]])

a is 211
tensor([[5.3519]])
tensor([[5.2950]])

a is 221
tensor([[5.3982]])
tensor([[5.3397]])

a is 231
tensor([[5.4424]])
tensor([[5.3849]])

a is 241
tensor([[5.4848]])
tensor([[5.4261]])

a is 251
tensor([[5.5255]